In [14]:
import requests
from bs4 import BeautifulSoup

url = "https://globalsummitguide.com/master-mountain-list"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

raw_names = []
mountains_list = soup.find_all('td', class_=None)

# find names in table (in span or a)
for td in mountains_list:
    a_tag = td.find('a')
    span_tag = td.find('span', class_='mtn-plain')
    
    if a_tag:
        raw_names.append(a_tag.get_text(strip=True))
    elif span_tag:
        raw_names.append(span_tag.get_text(strip=True))

# сlean from text in brackets
clean_names = []
for name in raw_names:
    if "(" in name:
        name = name.split("(")[0].strip()
    if name and name not in clean_names:
        clean_names.append(name)

print(f"Amount of collected mountains: {len(clean_names)}")
print(clean_names[:5])

Amount of collected mountains: 498
['Mount Everest', 'K2', 'Kangchenjunga', 'Lhotse', 'Makalu']


In [15]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

openapi_key=os.getenv("AI_API")



In [16]:
import random

# pool of structural styles to force syntactic diversity
style_pool = [
    "use passive voice at least once",
    "write one sentence as a direct quote from a person",
    "make one sentence look like an old historical record or archive text",
    "include other geographic features like rivers, valleys, or cities in the same text",
    "use complex punctuation (dashes, semicolons, or parenthetical remarks)",
    "start at least one sentence with a subordinate clause (e.g., 'although...', 'while...')",
    "make one sentence look like a technical scientific report about geology",
    "ensure the mountain name appears in the exact middle or end of the text, not just the beginning",
]

# list of corporate environments for negatives
negative_contexts = [
    "gaming/video game titles",
    "luxury watch and fountain pen manufacturing",
    "venture capital, asset management, and investment funds",
    "streetwear, outdoor clothing lines, and fashion brands",
    "cloud computing, infrastructure software, and database tech startups",
    "local artisanal cafes, bistros, and roasteries",
]

def get_dynamic_prompt(mountain_batch):
    mountains_str = ", ".join(mountain_batch)

    # pick random constraints for this specific batch loop
    selected_styles = random.sample(style_pool, k=2)
    selected_negatives = random.sample(negative_contexts, k=2)

    # assemble the ultimate prompt
    prompt = f"""
    act as an expert computational linguist and gold-standard ner data annotator.
    your task is to generate a highly diverse, non-templated dataset for named entity recognition (class: MOUNTAIN).
    
    target entities for this batch: [{mountains_str}]
    
    generation requirements:
    1. for each entity, generate exactly 2 positive sentences (where it is a real mountain) and 2 hard negative sentences (where it is NOT a mountain).
    2. structural diversity constraints for this batch (apply them across your answers):
       - {selected_styles[0]}
       - {selected_styles[1]}
    3. negative context constraints for this batch:
       - use these specific domains for corporate/product naming traps: {selected_negatives[0]} or {selected_negatives[1]}.
    4. linguistic rule: strictly avoid generic clichés like "the view from the base camp is breathtaking", "reached the summit at dawn", or simply "[mountain] logistics". make sentences natural, complex, and human-written.
    5. нou are absolutely forbidden to write anything other than the json of the response, neither at the beginning nor at the end of the response, you start and end your response exclusively with a dataset without any unnecessary phrases.
    output format constraints:
    - return only a valid json array of objects.
    - do not wrap the response in markdown code fences (```json).
    - character offsets ("start" and "end") must be perfectly accurate and 0-indexed.
    - for negative examples, the "entities" array have to be strictly empty [].
    
    json schema definition:
    [
      {{
        "text": "string content",
        "entities": [
          {{"start": int, "end": int, "label": "MOUNTAIN"}}
        ]
      }}
    ]
    """
    return prompt


In [ ]:
import json
import time
import google.generativeai as gemini

gemini.configure(api_key=openapi_key)

def generate_dataset_from_chatgpt(mountain_batch):
    mountains_str = ", ".join(mountain_batch)

    prompt = get_dynamic_prompt(mountains_str)
    
    try:
         model = gemini.GenerativeModel(
             "gemini-3.1-flash-lite",
             generation_config={"response_mime_type": "application/json"},
         )
         response = model.generate_content(prompt)
         return json.loads(response.text)

    except Exception as e:
        print(f"failed batch {mountain_batch}: {e}")
        return []

In [21]:
import json
import google.generativeai as gemini

gemini.configure(api_key=openapi_key)

def generate_dataset_from_chatgpt(mountain_batch):
    mountains_str = ", ".join(mountain_batch)

    prompt = get_dynamic_prompt(mountains_str)
    
    try:
         model = gemini.GenerativeModel(
             "gemini-3.1-flash-lite",
             generation_config={"response_mime_type": "application/json"},
         )
         response = model.generate_content(prompt)
         return json.loads(response.text)

    except Exception as e:
        print(f"failed batch {mountain_batch}: {e}")
        return []

In [22]:
dataset = []

for i in range(0, 352, 8):
    answer=generate_dataset_from_chatgpt(clean_names[i:i+8])
    dataset.extend(answer)

KeyboardInterrupt: 

In [ ]:
# fixing index shift issue

fixed_dataset=[]
with open("dataset.json", "r", encoding="utf-8") as f:
    broken_dataset = json.load(f)

fixed_dataset = []

for i, item in enumerate(broken_dataset):
    # every 3rd and 4th sentence in the pattern of 4 must be a hard negative
    if i % 4 >= 2:
        # completely strip the entity because it is a brand or product context
        item["entities"] = []
    
    fixed_dataset.append(item)

# save the final true-negative balanced dataset
with open("final_gold_dataset.json", "w", encoding="utf-8") as f:
    json.dump(fixed_dataset, f, ensure_ascii=False, indent=2)


In [ ]:
# load your current file to check the slices
with open("final_gold_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

print("--- validation report ---")
for i, item in enumerate(dataset):
    text = item["text"]
    entities = item["entities"]
    
    if entities:
        start = entities[0]["start"]
        end = entities[0]["end"]
        # pulling the actual string that lives under these indices
        actual_substring = text[start:end]

        if actual_substring not in clean_names: 
            print(f"line {i} | indices: {start}:{end} | actual text inside: '{actual_substring}'")